<a href="https://colab.research.google.com/github/yumimint/family_tree/blob/master/Family_Tree_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 家系図生成スクリプト

- これは、Googleスプレッドシートを読んで家系図のPDFを出力するPythonスクリプトです。
- 利用にあたりGoogleアカウントが必要です。
- 家系図PDFは[Googleドライブ](https://drive.google.com/drive/my-drive)に出力生成されます。

## 使い方

1. (初回のみ) 画面上部のメニューから 「ファイル」→「ドライブにコピーを保存」 をクリックします。

   これで、あなた自身のGoogleドライブにファイルがコピーされ、編集や保存ができるようになります。

2. あなた自身の家系データを用意します。

    サンプルデータとして[サザエさん家系図](https://docs.google.com/spreadsheets/d/1JL5Embx3XcSyUEQg8N8ty_6rdFh-DU0Fs3LpXkrniRA)と[ジョジョ家系図](https://docs.google.com/spreadsheets/d/1N7ROaHLjxNL2FBRNHN4ppsvahrFzs_TN04y-50PGPqM)があります。養子を記述するならジョジョが参考になるでしょう。

3. この下にある[データ定義](#データ定義)にご自身のスプレッドシートのURLを定義します。

    サンプルの行は頭に`#`を入れてコメント化する、または行ごと削除します。

4. 画面上部のメニューから「ランタイム」→「すべてのセルを実行」をクリックします。

    途中認証を求められるので適宜許可してやってください。まもなくPDFが出来ますので出来栄えを確認しましょう。

5. 問題があればデータを修正して 3. へ戻る。


## データ定義
スプレッドシートのURLを記述します。複数指定できます。

In [ ]:
OUTDIR = "Family Tree Example"
DATA = [
    # サザエさん家系図
    "https://docs.google.com/spreadsheets/d/1JL5Embx3XcSyUEQg8N8ty_6rdFh-DU0Fs3LpXkrniRA",

    # ジョジョ家系図
    "https://docs.google.com/spreadsheets/d/1N7ROaHLjxNL2FBRNHN4ppsvahrFzs_TN04y-50PGPqM",
]


## 日本語フォントと家系図生成ライブラリをインストール

In [ ]:
!apt-get -y install fonts-ipafont-gothic
try:
    from family_tree import Family, make_pdf
except ImportError:
    %pip install git+https://github.com/yumimint/family_tree.git
    from family_tree import Family, make_pdf


## Googleドライブの準備

In [ ]:
from pathlib import Path
import os
from google.colab import drive
drive.mount('/content/drive')
outdir = Path('/content/drive/MyDrive') / OUTDIR
outdir.mkdir(exist_ok=True, parents=True)
os.chdir(outdir)


## スプレッドシートを読み込んで家系図を生成する

In [ ]:
import gspread
import google.auth
from google.colab import auth

auth.authenticate_user()
creds, _ = google.auth.default()
client: gspread.Client = gspread.authorize(creds)

def get_from_sheet(client: gspread.Client, data: str):
    if data.startswith('https://'):
        ss = client.open_by_url(data)
    else:
        ss = client.open_by_key(data)
    print(ss.title)
    lines = []
    for worksheet in ss.worksheets():
        if worksheet.title in ["ノート"]:
            continue
        print(worksheet.title)
        rows = worksheet.get_all_values()
        # 2行目以降のA列とB列を取り出す
        lines.extend([
            "\t".join(row[:2]) + "\n"
            for row in rows[1:]
        ])
        # 空行を挿入して家族リストがシートを跨いで連結されるのを防ぐ
        lines.append("\n")
    return ss.title, lines, ss.lastUpdateTime

try:
    print("-------- 処理開始 --------")
    whole_f = Family()
    whole_lastUpdateTime = ""

    for data in DATA:
        title, lines, lastUpdateTime = get_from_sheet(client, data)
        whole_lastUpdateTime = max(whole_lastUpdateTime, lastUpdateTime)

        f = Family()
        whole_f.populate(lines)
        f.populate(lines)
        f.postprocess()
        make_pdf(f, title, label=f"{title} {lastUpdateTime}")

    if len(DATA) > 1:
        whole_f.postprocess()
        title = "全体家系図"
        make_pdf(whole_f, title, label=f"{title} {whole_lastUpdateTime}")
    print("-------- 処理完了 --------")

except Exception as e:
    print(f"エラーが発生しました：{e}")
